# RAG Pipeline

In [55]:
from langchain_core.documents import Document

In [56]:
doc = Document(
    page_content = "This is a sample page content to learn the 1st and very important step - Data",
    metadata = {
        "source": "example.txt",
        "pages": 1,
        "author": "Ashwani",
        "date-created": "2026-05-10"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Ashwani', 'date-created': '2026-05-10'}, page_content='This is a sample page content to learn the 1st and very important step - Data')

## Ingestion Pipeline

### Fetch and load PDF files

In [57]:
!pip install langchain_community langchain-text-splitters pypdf

In [58]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [59]:
### Read all the pdf's inside the directory
def processAllPdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    print(pdf_dir)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = processAllPdfs("data")

data
Found 2 PDF files to process

Processing: TCS_Earning_Call_Transcript.pdf
  ✓ Loaded 35 pages

Processing: Amazon_Bedrock_User_Guide.pdf
  ✓ Loaded 9 pages

Total documents loaded: 44


In [60]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 26.1.163', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-14T19:49:38+05:30', 'author': 'Parizad  Khokhri', 'moddate': '2026-04-14T19:57:44+05:30', 'title': '', 'source': 'data/TCS_Earning_Call_Transcript.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_file': 'TCS_Earning_Call_Transcript.pdf', 'file_type': 'pdf'}, page_content='9th Floor Nirmal Building Nariman Point Mumbai 400 021 \nTel 91 22 6778 9595 Fax 91 22 6630 3672 e-mail corporate.office@tcs.com website www.tcs.com \nRegistered Office 9th Floor Nirmal Building Nariman Point Mumbai 400 021 \nCorporate Identity No. (CIN): L22210MH1995PLC084781 \nTCS/SE/10/2026-27 \n \nApril 14, 2026 \n \nNational Stock Exchange of India Limited  BSE Limited \nExchange Plaza, C-1, Block G,    P. J. Towers,       \nBandra Kurla Complex, Bandra (East)    Dalal Street, \nMumbai - 400051                                                             Mumbai - 400001   

### Split Text into chunks


In [61]:
def splitDocuments(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [62]:
chunks=splitDocuments(all_pdf_documents)
chunks

Split 44 documents into 97 chunks

Example chunk:
Content: 9th Floor Nirmal Building Nariman Point Mumbai 400 021 
Tel 91 22 6778 9595 Fax 91 22 6630 3672 e-mail corporate.office@tcs.com website www.tcs.com 
Registered Office 9th Floor Nirmal Building Nariman...
Metadata: {'producer': 'Adobe PDF Library 26.1.163', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-14T19:49:38+05:30', 'author': 'Parizad  Khokhri', 'moddate': '2026-04-14T19:57:44+05:30', 'title': '', 'source': 'data/TCS_Earning_Call_Transcript.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_file': 'TCS_Earning_Call_Transcript.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 26.1.163', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-14T19:49:38+05:30', 'author': 'Parizad  Khokhri', 'moddate': '2026-04-14T19:57:44+05:30', 'title': '', 'source': 'data/TCS_Earning_Call_Transcript.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_file': 'TCS_Earning_Call_Transcript.pdf', 'file_type': 'pdf'}, page_content='9th Floor Nirmal Building Nariman Point Mumbai 400 021 \nTel 91 22 6778 9595 Fax 91 22 6630 3672 e-mail corporate.office@tcs.com website www.tcs.com \nRegistered Office 9th Floor Nirmal Building Nariman Point Mumbai 400 021 \nCorporate Identity No. (CIN): L22210MH1995PLC084781 \nTCS/SE/10/2026-27 \n \nApril 14, 2026 \n \nNational Stock Exchange of India Limited  BSE Limited \nExchange Plaza, C-1, Block G,    P. J. Towers,       \nBandra Kurla Complex, Bandra (East)    Dalal Street, \nMumbai - 400051                                                             Mumbai - 400001   

### Create embeddings And store them in Vector DB

In [63]:
!pip install chromadb

In [64]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [65]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: embeddings model name
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generateEmbeddings(self, chunks: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of chunks

        Args:
            chunks: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(chunks), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(chunks)} chunks...")
        embeddings = self.model.encode(chunks, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_15068/1797790183.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Store Vectors

In [80]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store_1"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def addDocuments(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


### Let's now convert our chunk of text into embeddings

In [81]:
chunk_list=[doc.page_content for doc in chunks]

In [82]:
embeddings=embedding_manager.generateEmbeddings(chunk_list)

Generating embeddings for 97 chunks...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Generated embeddings with shape: (97, 384)


In [83]:
vector_store.addDocuments(chunks, embeddings)

Adding 97 documents to vector store...
Successfully added 97 documents to vector store
Total documents in collection: 97


## Retrieval pipeline From Vector DB

In [84]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generateEmbeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store,embedding_manager)
rag_retriever

In [85]:
rag_retriever.retrieve("tell me about TCS Partnerships")

Retrieving documents for query: 'tell me about TCS Partnerships'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ddfc6d80_37',
  'content': "expand into adjacent businesses while simultaneously enhancing the \nefficiency of their core operations. This strategic shift is paving the way \nfor growth and innovation. \no Momentum remains strong in securing large deals within the \ntelecommunications sector. We have signed our ﬁrst mega deal in \nCMI this quarter - a signiﬁcant expansion of its long-standing \npartnership with a leading UK-based telecom operator. This ﬁve-year \ncontract will see TCS lead the operator's comprehensive IT \ntransformation journey for its consumer business, leveraging advanced \nAI, Cloud, and Digital Engineering capabilities. The expansion, built \non the strength of years of trusted partnership and demonstrated \ndelivery, will see TCS take end-to-end responsibility for running the \nentire IT systems of the operator's consumer base and become its \nstrategic technology partner, consolidating the entire landscape \npreviously spread across multiple service

In [86]:
rag_retriever.retrieve("What is a prompt?")

Retrieving documents for query: 'What is a prompt?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0f40e169_94',
  'content': "the model must respond with the correct choice. An example classiﬁcation use case is sentiment \nanalysis: the input is a text passage, and the model must classify the sentiment of the text, such \nas whether it's positive or negative, or harmless or toxic.\n• Question-answer, without context: The model must answer the question with its internal \nknowledge without any context or document.\n• Question-answer, with context: The user provides an input text with a question, and the model \nmust answer the question based on information provided within the input text.\nWhat is prompt engineering? 194",
  'metadata': {'content_length': 595,
   'page': 7,
   'file_type': 'pdf',
   'source': 'data/Amazon_Bedrock_User_Guide.pdf',
   'source_file': 'Amazon_Bedrock_User_Guide.pdf',
   'total_pages': 9,
   'creationdate': 'D:20240409110030',
   'page_label': '8',
   'producer': 'PDFium',
   'doc_index': 94,
   'creator': 'PDFium'},
  'similarity_score': 0.1

## RAG Pipeline- VectorDB To LLM Output Generation

In [87]:
from google.colab import userdata

In [88]:
!pip install -U langchain langchain-google-genai

In [89]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

In [90]:
class GenAI:
  def __init__(self, model_name: str = "gemini-2.5-flash-lite", api_key: str =None):
        """
        Initialize gemini-2.5-flash-lite

        Args:
            model_name: Gemini model name (Gemini 3.1 Flash-Lite, Gemini 3.1 Pro, etc.)
            api_key: Gemini API key (or set Gemini environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or userdata.get('Gemini_API')

        if not self.api_key:
            raise ValueError("Gemini API key is required. Set Gemini_API environment variable or pass api_key parameter.")

        self.llm = ChatGoogleGenerativeAI(
            google_api_key=self.api_key,
            model=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Gemini LLM with model: {self.model_name}")

  def generateResponse(self, query: str, context: str, max_length: int = 500) -> str:
    """
      Generate response using retrieved context

      Args:
          query: User question
          context: Retrieved document context
          max_length: Maximum response length

      Returns:
          Generated response string
    """

    # Create prompt template
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely. Context: {context} Question: {question} Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
    )

    # Format the prompt
    formatted_prompt = prompt_template.format(context=context, question=query)

    try:
        # Generate response
        # messages = [HumanMessage(content=formatted_prompt)]
        response = self.llm.invoke(formatted_prompt)
        return response.content

    except Exception as e:
        return f"Error generating response: {str(e)}"

  def generateSimpleResponse(self, query: str, context: str) -> str:
      """
      Simple response generation without complex prompting

      Args:
          query: User question
          context: Retrieved context

      Returns:
          Generated response
      """
      simple_prompt = f"""Based on this context: {context} Question: {query} Answer:"""

      try:
          messages = [HumanMessage(content=simple_prompt)]
          response = self.llm.invoke(messages)
          return response.content
      except Exception as e:
          return f"Error: {str(e)}"

### Initialize Gemini LLM (you'll need to set Gemini_API key environment variable)

In [91]:
try:
    gemini_llm = GenAI()
    print("Gemini LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your Gemini_API key environment variable to use the LLM.")
    gemini_llm = None

Initialized Gemini LLM with model: gemini-2.5-flash-lite
Gemini LLM initialized successfully!


In [51]:
rag_retriever.retrieve("How to design your prompts")

Retrieving documents for query: 'How to design your prompts'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_cded37f4_93',
  'content': 'Prompt engineering refers to the practice of crafting and optimizing input prompts by selecting \nappropriate words, phrases, sentences, punctuation, and separator characters to eﬀectively \nuse LLMs for a wide variety of applications. In other words, prompt engineering is the art of \ncommunicating with an LLM. High-quality prompts condition the LLM to generate desired or better \nresponses. The detailed guidance provided within this document is applicable across all LLMs \nwithin Amazon Bedrock.\nThe best prompt engineering approach for your use case is dependent on both the task and the \ndata. Common tasks supported by LLMs on Amazon Bedrock include:\n• Classiﬁcation: The prompt includes a question with several possible choices for the answer, and \nthe model must respond with the correct choice. An example classiﬁcation use case is sentiment \nanalysis: the input is a text passage, and the model must classify the sentiment of the text, such

### Integration Vector DB Context pipeline With LLM output

In [92]:
def ragWithLLM(query, retriever, llm, top_k = 3, return_context=False):
  ## retriever the context
  context = retriever.retrieve(query, top_k = top_k)
  ## Call LLM with top_k chunks
  response = llm.generateResponse(query = query, context = context)
  ## return the final response from LLM
  return response

In [93]:
answer=ragWithLLM("What is a prompt?", rag_retriever, gemini_llm)
print(answer)

Retrieving documents for query: 'What is a prompt?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Prompts are specific inputs provided by the user that guide Large Language Models (LLMs) to generate an appropriate response or output for a given task or instruction.
